# BEST-EFFORT ROUND 0 A100 Gaussian splat recovery

This fixed output-first notebook restores the complete Round 0 cache for the same audited selection. First complete `learned_quality_cache_audit.ipynb` and confirm `TRACK AUDIT PASSED`. Then Choose an **A100 High-RAM** runtime and use **Runtime → Run all**. This transparent best-effort experiment keeps the legacy result untouched, publishes `<input>_learned_test_result`, and includes diagnostics—not a web viewer.

In [ ]:
INPUT_FOLDER = ""  # @param {type:"string"}


In [ ]:
import json, shutil, subprocess
gpu_line = subprocess.run([
    'nvidia-smi', '--query-gpu=name,memory.total',
    '--format=csv,noheader,nounits',
], check=True, capture_output=True, text=True).stdout.splitlines()[0]
gpu_name, memory_mib = (part.strip() for part in gpu_line.rsplit(',', 1))
vram_gib = float(memory_mib) / 1024.0
assert 'A100' in gpu_name.upper(), f'A100 required; detected {gpu_name}'
assert vram_gib >= 39.0, f'At least 39 GiB VRAM required; detected {vram_gib:.1f}'
disk_gib = shutil.disk_usage('/content').free / (1024 ** 3)
assert disk_gib >= 80.0, f'At least 80 GiB local disk required; detected {disk_gib:.1f}'
print(json.dumps({'gpu': gpu_name, 'vram_gib': round(vram_gib, 1), 'disk_free_gib': round(disk_gib, 1)}, sort_keys=True))


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive').resolve()


In [ ]:
import json, unicodedata
from pathlib import Path, PurePosixPath
raw_folder = INPUT_FOLDER.strip()
if not raw_folder:
    raw_folder = input('MyDrive-relative input folder: ').strip()
assert raw_folder and '\\' not in raw_folder, 'Use a MyDrive-relative POSIX path'
assert not any(unicodedata.category(ch) == 'Cc' for ch in raw_folder)
folder = PurePosixPath(raw_folder)
assert not folder.is_absolute() and folder.parts
assert all(part not in {'', '.', '..'} for part in folder.parts)
assert not raw_folder.endswith(('_result', '_learned_test_result', '_learned_test_diagnostics', '_learned_test_cache'))
INPUT_PATH = DRIVE_ROOT.joinpath(*folder.parts).resolve()
INPUT_PATH.relative_to(DRIVE_ROOT)
assert INPUT_PATH.is_dir(), f'Input folder does not exist: {INPUT_PATH}'
RESULT_PATH = INPUT_PATH.with_name(INPUT_PATH.name + '_learned_test_result')
CACHE_PATH = INPUT_PATH.with_name(INPUT_PATH.name + '_learned_test_cache')
RUN_SPEC = {'schema_version': 1, 'input_folder': folder.as_posix(), 'recovery_mode': 'round0_output_first_v1', 'publish': {'replace_owned_result': True}}
SPEC_PATH = Path('/content/learned_spec.json')
with SPEC_PATH.open('w', encoding='utf-8') as handle:
    json.dump(RUN_SPEC, handle, sort_keys=True, separators=(',', ':'))
print(f'Input: {INPUT_PATH}')
print(f'Pre-training cache: {CACHE_PATH}')
print(f'Result: {RESULT_PATH}')


In [ ]:
import shutil, subprocess
from pathlib import Path
SOURCE_ROOT = Path('/content/gaussian-splatter-src')
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
REPOSITORY_URL = 'https://github.com/mehmettahacumurcu/gaussian-splatter.git'
COMMIT_SHA = '982c6a1d487cee5e28eb855ed8f8fcb38f5cdb52'
subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(SOURCE_ROOT)], check=True)
subprocess.run(['git', '-C', str(SOURCE_ROOT), 'checkout', '--detach', COMMIT_SHA], check=True)
actual = subprocess.run(['git', '-C', str(SOURCE_ROOT), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
assert actual == COMMIT_SHA, 'Immutable source checkout mismatch'


In [ ]:
import subprocess
subprocess.run(['bash', 'colab/static_notebook_bootstrap.sh'], cwd=SOURCE_ROOT, check=True)


In [ ]:
import sys
sys.path.insert(0, str(SOURCE_ROOT))
from huggingface_hub import snapshot_download
from experiments.learned_quality.dependencies import (
    install_learned_environment, materialize_pinned_assets,
)
def resolved_revision(local_path):
    metadata_root = Path(local_path) / '.cache' / 'huggingface' / 'download'
    revisions = set()
    for metadata in metadata_root.rglob('*.metadata'):
        first = metadata.read_text(encoding='utf-8').splitlines()[0].strip()
        if len(first) == 40:
            revisions.add(first)
    assert len(revisions) == 1, f'Cannot authenticate checkpoint: {local_path}'
    return revisions.pop()
LEARNED_ENV = install_learned_environment(Path(sys.executable).resolve())
LEARNED_ASSETS = materialize_pinned_assets(
    LEARNED_ENV, downloader=snapshot_download, resolve_revision=resolved_revision,
)


In [ ]:
from experiments.learned_quality.dependencies import verify_learned_environment
MODEL_MANIFEST = verify_learned_environment(LEARNED_ENV, LEARNED_ASSETS)
assert MODEL_MANIFEST.path == Path('/content/learned-env/model_manifest.json')
print(f'Verified model manifest: {MODEL_MANIFEST.path}')


In [ ]:
import json, os, subprocess, traceback
from pathlib import Path
from google.colab import drive, runtime
environment = dict(os.environ)
environment['LEARNED_MODEL_MANIFEST'] = str(MODEL_MANIFEST.path)
environment['PYTHONUNBUFFERED'] = '1'
failure = None
try:
    completed = subprocess.run(
        [
            "/content/learned-env/bin/python",
            "-u",
            "-m",
            "scripts.learned_quality_run",
            "--spec",
            "/content/learned_spec.json",
        ],
        cwd=SOURCE_ROOT,
        env=environment,
        check=False,
    )
    receipt_path = Path('/content/learned_run_result.json')
    if not receipt_path.is_file():
        raise RuntimeError('Training ended without a run receipt')
    receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
    run_succeeded = (
        completed.returncode == 0 and receipt.get('status') == 'success'
    )
    if not run_succeeded:
        diagnostics = receipt.get('diagnostics_path')
        if diagnostics:
            print(f'Failure diagnostics: {diagnostics}')
        raise RuntimeError(f'Learned-quality run failed: {receipt}')
    assert RESULT_PATH.name.endswith('_learned_test_result')
    expected = RESULT_PATH.resolve()
    actual = Path(receipt['final_path']).resolve()
    if actual != expected:
        raise RuntimeError(f'Unexpected result path: {actual}')
    success = json.loads(
        (actual / '_SUCCESS').read_text(encoding='utf-8')
    )
    if success.get('run_id') != receipt.get('run_id'):
        raise RuntimeError('Published success marker does not match this run')
    required = (
        'splat.ply', 'quality_report.json', 'experiment_report.json',
        'diagnostics/masks_contact_sheet.png',
        'diagnostics/depth_contact_sheet.png',
        'diagnostics/geometry_contact_sheet.png',
        'diagnostics/final_render_contact_sheet.png',
    )
    missing = [relative for relative in required if not (actual / relative).is_file()]
    if missing:
        raise RuntimeError(f'Published result is incomplete: {missing}')
    print(f'Result folder: {actual}')
    print(f'Splat: {actual / "splat.ply"}')
    print(f'Experiment report: {actual / "experiment_report.json"}')
    print('Training and Drive publication completed successfully.')
except BaseException as exc:
    failure = exc
    print(f'Run ended with {type(exc).__name__}: {exc}')
    traceback.print_exception(type(exc), exc, exc.__traceback__)
finally:
    print('Flushing outstanding Google Drive writes...')
    try:
        drive.flush_and_unmount()
    except BaseException as flush_error:
        print(f'Drive flush/unmount failed: {flush_error}')
    print('Releasing the Colab runtime now.')
    try:
        runtime.unassign()
    except BaseException as release_error:
        print(f'Runtime release request failed: {release_error}')
        if failure is None:
            failure = release_error
if failure is not None:
    raise failure
